# Module 08, Unit 03 — Posterior Analysis & Course Retrospective

> **Threat Surfaces: Multivariable Calculus for AI Security**
> fischer³ Education | Module 08 | Unit 03

| | |
|---|---|
| **Estimated time** | 60–70 min |
| **Exercises** | Download PDF from the course repository |
| **Statistical threads** | All four: `[PD]` · `[MLE]` · `[BAY]` · `[GLM]` |
| **Prerequisites** | Module 08 Units 01–02 |

---

## Learning Objectives

By the end of this unit you will be able to:

- [ ] Compute the posterior predictive distribution via Monte Carlo integration
- [ ] Visualize decision boundaries with uncertainty bands from the variational posterior
- [ ] Compute and interpret a calibration curve
- [ ] Compare the variational solution to the MAP and MLE solutions
- [ ] Identify which tool from each module appears in the Bayesian logistic regression derivation

---

## What We Have Built

At the end of Unit 02, we obtained a variational posterior $q^*(\boldsymbol{w}) = \mathcal{N}(\boldsymbol{m}^*, \text{diag}((\boldsymbol{s}^*)^2))$ over the logistic regression weights. This unit extracts everything useful from that posterior:

- **Predictions with uncertainty**: not just a class probability, but a distribution over class probabilities
- **Decision boundary with confidence region**: where is the boundary, and how wide is the uncertainty?
- **Comparison to simpler methods**: what does variational inference buy us over MAP or MLE?
- **Course retrospective**: which mathematical tool from each module made this possible?

In [ ]:
import subprocess, sys
for pkg in ["numpy", "matplotlib", "scipy", "scikit-learn"]:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
print("All packages installed.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from scipy import stats

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.figsize': (9,6), 'font.size': 11,
                     'axes.titlesize': 13, 'lines.linewidth': 2, 'figure.dpi': 120})
TS_BLUE='#1e64b4'; TS_AMBER='#c87814'; TS_GREEN='#1e8c50'
TS_GRAY='#64646e'; TS_RED='#b41e1e'; TS_LIGHT='#f5f7fa'

# ── Reproduce dataset and all functions from Units 01-02 ─────
iris = load_iris()
mask = iris.target < 2
X_raw = iris.data[mask, :2]
y = iris.target[mask].astype(float)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)
n = len(y)
X = np.column_stack([np.ones(n), X_scaled])
p_dim = X.shape[1]

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-np.clip(z, -500, 500)))

def log_likelihood(w, X, y):
    p_hat = np.clip(sigmoid(X @ w), 1e-12, 1-1e-12)
    return np.sum(y * np.log(p_hat) + (1-y) * np.log(1-p_hat))

def grad_log_likelihood(w, X, y):
    return X.T @ (y - sigmoid(X @ w))

def kl_divergence(m, log_s, sigma0_sq):
    s_sq = np.exp(2 * log_s)
    return 0.5 * np.sum(s_sq/sigma0_sq + m**2/sigma0_sq - 1 - np.log(s_sq/sigma0_sq))

def grad_kl(m, log_s, sigma0_sq):
    s_sq = np.exp(2 * log_s)
    return m/sigma0_sq, s_sq/sigma0_sq - 1.0

def elbo_and_grad(m, log_s, X, y, sigma0_sq, K=10, rng=None):
    if rng is None: rng = np.random.default_rng()
    s = np.exp(log_s)
    eps = rng.standard_normal((K, len(m)))
    w_samples = m + s * eps
    ll_vals  = np.array([log_likelihood(w_samples[k], X, y) for k in range(K)])
    g_ll_w   = np.array([grad_log_likelihood(w_samples[k], X, y) for k in range(K)])
    g_ll_m   = g_ll_w.mean(axis=0)
    g_ll_rho = (s * eps * g_ll_w).mean(axis=0)
    kl = kl_divergence(m, log_s, sigma0_sq)
    g_kl_m, g_kl_rho = grad_kl(m, log_s, sigma0_sq)
    return ll_vals.mean() - kl, g_ll_m - g_kl_m, g_ll_rho - g_kl_rho

def run_vi(X, y, sigma0_sq=1.0, lr=0.05, n_iter=600, K=10, seed=42):
    rng = np.random.default_rng(seed)
    p = X.shape[1]
    m = np.zeros(p); log_s = np.zeros(p)
    elbo_history = []
    for t in range(n_iter):
        elbo, gm, gr = elbo_and_grad(m, log_s, X, y, sigma0_sq, K=K, rng=rng)
        m += lr * gm; log_s += lr * gr
        elbo_history.append(elbo)
    return {'m': m, 'log_s': log_s, 's': np.exp(log_s), 'elbo': np.array(elbo_history)}

# Run VI to get the posterior
vi_results = run_vi(X, y, sigma0_sq=1.0, lr=0.05, n_iter=600, K=10)
m_star  = vi_results['m']
s_star  = vi_results['s']
print(f'Posterior mean m* = {np.round(m_star, 4)}')
print(f'Posterior std  s* = {np.round(s_star, 4)}')

---

## 1. The Posterior Predictive Distribution

For a new input $\boldsymbol{x}^*$, the Bayesian prediction is not a single class probability but a **distribution** over class probabilities:

$$
p(y^* = 1 \mid \boldsymbol{x}^*, \boldsymbol{X}, \boldsymbol{y}) = \int \sigma(\boldsymbol{w}^{\top}\boldsymbol{x}^*)\,p(\boldsymbol{w} \mid \boldsymbol{X}, \boldsymbol{y})\,d\boldsymbol{w}
$$

We approximate by replacing $p(\boldsymbol{w} \mid \boldsymbol{X}, \boldsymbol{y})$ with $q^*(\boldsymbol{w})$ and using Monte Carlo:

$$
\hat{P}(y^* = 1 \mid \boldsymbol{x}^*) \approx \frac{1}{K}\sum_{k=1}^K \sigma(\boldsymbol{w}^{(k)\top}\boldsymbol{x}^*), \qquad \boldsymbol{w}^{(k)} \sim q^*(\boldsymbol{w})
$$

The spread of the sampled probabilities $\{\sigma(\boldsymbol{w}^{(k)\top}\boldsymbol{x}^*)\}$ quantifies our **epistemic uncertainty** — uncertainty about the model parameters rather than inherent class overlap.

In [ ]:
# ============================================================
# Section 1 — Posterior predictive distribution
# ============================================================
rng = np.random.default_rng(0)
N_pred = 2000   # posterior samples for prediction

# Draw weight samples from q*(w) = N(m*, diag((s*)²))
eps_pred = rng.standard_normal((N_pred, p_dim))
w_post   = m_star + s_star * eps_pred   # (N_pred, p_dim)

# Posterior predictive on the training set
# p_hat[k, i] = σ(w^(k)ᵀ xᵢ)
logits = X @ w_post.T               # (n, N_pred)
p_post = sigmoid(logits)            # (n, N_pred)

# Mean and std of predictive distribution per observation
p_mean = p_post.mean(axis=1)       # (n,)
p_std  = p_post.std(axis=1)        # (n,) — epistemic uncertainty

# Accuracy using posterior mean prediction
y_pred = (p_mean >= 0.5).astype(float)
acc    = (y_pred == y).mean()

print(f'Posterior predictive accuracy: {acc:.1%}')
print(f'Mean predictive uncertainty (std): {p_std.mean():.4f}')
print(f'Max predictive uncertainty:        {p_std.max():.4f}  (most uncertain observation)')

# Visualize uncertainty on the training set
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
sort_idx = np.argsort(p_mean)
ax.errorbar(range(n), p_mean[sort_idx], yerr=p_std[sort_idx],
            fmt='o', color=TS_BLUE, alpha=0.6, markersize=3,
            ecolor=TS_GRAY, elinewidth=0.8, capsize=0)
ax.scatter(range(n), y[sort_idx], marker='|', color=TS_AMBER, s=60,
           linewidths=1.5, label='True label')
ax.axhline(0.5, color=TS_RED, lw=1.2, ls='--', label='Decision boundary')
ax.set_xlabel('Observation (sorted by predicted probability)')
ax.set_ylabel('Predicted probability')
ax.set_title('Posterior predictive mean ± std\n(vertical bars = true labels)')
ax.legend(fontsize=9)

ax2 = axes[1]
ax2.scatter(p_mean, p_std, c=y, cmap='coolwarm', alpha=0.7, s=45,
            edgecolors='white', linewidths=0.5)
ax2.set_xlabel('Posterior predictive mean $\\hat{p}$')
ax2.set_ylabel('Posterior predictive std')
ax2.set_title('Uncertainty is highest near $\\hat{p} = 0.5$\n(decision boundary)')

plt.suptitle('Posterior predictive distribution — uncertainty from $q^*(\\mathbf{w})$',
             fontsize=11, y=1.01)
plt.tight_layout()
plt.show()

**What do you see?** Epistemic uncertainty is highest near the decision boundary ($\hat{p} \approx 0.5$) where the model is most unsure, and lowest in the regions far from the boundary where the classes are clearly separated. This is the qualitative behavior we expect from a well-calibrated Bayesian model.

---

## 2. Decision Boundary with Uncertainty Bands

In [ ]:
# ============================================================
# Section 2 — Decision boundary with uncertainty bands
# Compare: VI (Bayesian), MAP (Ridge), MLE (no regularization)
# ============================================================

# ── MAP / MLE via sklearn (for comparison) ───────────────────
lr_mle = LogisticRegression(C=1e6, fit_intercept=False, solver='lbfgs')
lr_mle.fit(X, y)
w_mle = lr_mle.coef_[0]

# MAP: Ridge penalty λ = 1/σ₀² = 1.0  →  C = 1/λ = 1.0
lr_map = LogisticRegression(C=1.0, fit_intercept=False, solver='lbfgs')
lr_map.fit(X, y)
w_map = lr_map.coef_[0]

# ── Grid for plotting ─────────────────────────────────────────
x1g = np.linspace(-3.0, 3.5, 200)
x2g = np.linspace(-3.0, 3.0, 200)
X1g, X2g = np.meshgrid(x1g, x2g)
X_grid = np.column_stack([np.ones(X1g.ravel().shape), X1g.ravel(), X2g.ravel()])

# VI: posterior predictive mean and std on the grid
rng2 = np.random.default_rng(1)
eps_g = rng2.standard_normal((N_pred, p_dim))
w_g   = m_star + s_star * eps_g
p_grid_post = sigmoid(X_grid @ w_g.T).mean(axis=1).reshape(X1g.shape)
p_grid_std  = sigmoid(X_grid @ w_g.T).std(axis=1).reshape(X1g.shape)

# MLE and MAP on the grid
p_grid_mle = sigmoid(X_grid @ w_mle).reshape(X1g.shape)
p_grid_map = sigmoid(X_grid @ w_map).reshape(X1g.shape)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, p_grid, title, w_line in [
    (axes[0], p_grid_mle, 'MLE (no regularization)', w_mle),
    (axes[1], p_grid_map, 'MAP (Ridge, σ₀²=1)', w_map),
    (axes[2], p_grid_post, 'Variational Bayes (posterior mean)', m_star),
]:
    cf = ax.contourf(X1g, X2g, p_grid, levels=20,
                     cmap='RdBu_r', alpha=0.55, vmin=0, vmax=1)
    ax.contour(X1g, X2g, p_grid, levels=[0.5],
               colors=['black'], linewidths=2.0)

    for c, col, lab in [(0, TS_BLUE, 'Setosa'), (1, TS_AMBER, 'Versicolor')]:
        idx = y == c
        ax.scatter(X_scaled[idx,0], X_scaled[idx,1], color=col,
                   alpha=0.8, s=45, edgecolors='white', linewidths=0.5,
                   label=lab)

    ax.set_xlabel('Sepal length (std)'); ax.set_ylabel('Sepal width (std)')
    ax.set_title(title); ax.legend(fontsize=8)
    ax.set_xlim(-3, 3.5); ax.set_ylim(-3, 3)

# Add uncertainty contours to VI panel
axes[2].contour(X1g, X2g, p_grid_std, levels=[0.05, 0.10, 0.15],
                colors=[TS_GREEN], linewidths=1.2, linestyles='--',
                alpha=0.8)
axes[2].text(0.05, 0.92, 'Dashed = uncertainty\nbands (σ=0.05,0.10,0.15)',
             transform=axes[2].transAxes, fontsize=7, color=TS_GREEN,
             bbox=dict(facecolor=TS_LIGHT, alpha=0.85))

plt.colorbar(cf, ax=axes[2], label='P(Versicolor)')
plt.suptitle('Decision boundaries: MLE vs MAP vs Variational Bayes',
             fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

print(f'Accuracy: MLE={lr_mle.score(X,y):.1%}  MAP={lr_map.score(X,y):.1%}  VI={acc:.1%}')
print(f'\nWeight comparison:')
print(f'  MLE  w = {np.round(w_mle,4)}')
print(f'  MAP  w = {np.round(w_map,4)}')
print(f'  VI   m = {np.round(m_star,4)}')

**What do you see?** All three methods achieve similar accuracy on this well-separated problem. The key difference is in the uncertainty representation: MLE and MAP produce a single decision boundary with no uncertainty estimate; the variational posterior produces a mean boundary (solid black) surrounded by uncertainty contours (dashed green) that widen in the regions with fewer training points. The MAP weights are shrunk toward zero compared to MLE — the Ridge prior at work. The VI mean is close to MAP, confirming the connection $\hat{\boldsymbol{\beta}}_{\text{Ridge}} = \boldsymbol{m}^*_{\text{VI}}$ from Module 06 Unit 02.

---

## 3. Calibration and Mahalanobis Anomaly Detection

In [ ]:
# ============================================================
# Section 3 — Calibration and Mahalanobis anomaly detection
# ============================================================

# ── Calibration curve ────────────────────────────────────────
# Bin predicted probabilities and compare to empirical frequency
n_bins = 8
bin_edges = np.linspace(0, 1, n_bins+1)
bin_centers = 0.5*(bin_edges[:-1] + bin_edges[1:])

cal_mean_pred = []
cal_frac_pos  = []
for lo, hi in zip(bin_edges[:-1], bin_edges[1:]):
    in_bin = (p_mean >= lo) & (p_mean < hi)
    if in_bin.sum() > 0:
        cal_mean_pred.append(p_mean[in_bin].mean())
        cal_frac_pos.append(y[in_bin].mean())

# ── Mahalanobis anomaly detection in weight space ────────────
# Use the posterior q*(w) = N(m*, diag((s*)²)) as the "normal" model
# Mahalanobis distance of each w_post sample from m*
# d_M² = (w-m*)ᵀ Σ⁻¹ (w-m*) where Σ = diag((s*)²)
# For diagonal Σ: d_M² = Σⱼ (wⱼ - mⱼ*)²/sⱼ*²

def mahal_diag(w_samples, m, s):
    """Mahalanobis distance from posterior samples to mean."""
    return np.sqrt(np.sum(((w_samples - m)/s)**2, axis=1))

d_m_post = mahal_diag(w_post, m_star, s_star)
threshold_95 = np.sqrt(stats.chi2.ppf(0.95, df=p_dim))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: calibration curve
ax = axes[0]
ax.plot([0,1],[0,1], '--', color=TS_GRAY, lw=1.5, label='Perfect calibration')
ax.plot(cal_mean_pred, cal_frac_pos, 'o-', color=TS_BLUE, lw=2,
        markersize=8, label='Variational Bayes')
ax.fill_between(cal_mean_pred,
                np.array(cal_frac_pos) - 0.1,
                np.array(cal_frac_pos) + 0.1,
                alpha=0.12, color=TS_BLUE)
ax.set_xlim(0,1); ax.set_ylim(0,1)
ax.set_xlabel('Mean predicted probability')
ax.set_ylabel('Fraction of positives (empirical)')
ax.set_title('Calibration curve\n(diagonal = perfect calibration)')
ax.legend(fontsize=9)
ax.set_aspect('equal')

# Right: Mahalanobis distance distribution of posterior samples
ax2 = axes[1]
ax2.hist(d_m_post, bins=50, density=True, color=TS_BLUE, alpha=0.6,
         edgecolor='white', label='Posterior samples')

# Theoretical χ distribution (sqrt of χ² with p_dim df)
d_vals = np.linspace(0, 6, 200)
chi_pdf = stats.chi(df=p_dim).pdf(d_vals)
ax2.plot(d_vals, chi_pdf, color=TS_AMBER, lw=2.5,
         label=fr'$\chi_{{p}}$ distribution (p={p_dim})')
ax2.axvline(threshold_95, color=TS_RED, lw=2, ls='--',
            label=f'95% threshold = {threshold_95:.2f}')
ax2.set_xlabel('Mahalanobis distance $d_M$')
ax2.set_ylabel('Density')
ax2.set_title('Posterior weight samples: Mahalanobis distance\nfrom posterior mean')
ax2.legend(fontsize=9)

plt.suptitle('Calibration and posterior geometry', fontsize=11, y=1.01)
plt.tight_layout()
plt.show()

frac_outside = (d_m_post > threshold_95).mean()
print(f'Fraction of posterior samples outside 95% Mahalanobis ball: {frac_outside:.1%}  (expect ≈5%)')

**What do you see?**

- **Calibration**: the variational posterior is reasonably calibrated — predicted probabilities near the empirical frequencies. The slight deviation at the extremes is expected for a model with a finite prior.
- **Mahalanobis distance**: the posterior weight samples, when centered at $\boldsymbol{m}^*$ and scaled by $\boldsymbol{s}^*$, follow a $\chi_p$ distribution — the exact distribution of the Mahalanobis distance for samples from a Gaussian (Module 06 Unit 01, Section 3). The fraction outside the 95% threshold is approximately 5%, confirming the posterior is correctly calibrated as a multivariate Gaussian.

---

## 4. Course Retrospective — Every Module in One Table

The following table maps each module to its specific contribution to the Bayesian logistic regression derivation we completed in Module 08. Nothing in this derivation was ad hoc — every step used a tool built explicitly for this purpose.

| Module | Title | Contribution to Module 08 |
|---|---|---|
| **00** | Orientation & Prerequisites | Vectors and norms: $\boldsymbol{x}$, $\|\boldsymbol{w}\|^2$; sigmoid definition; integration by substitution |
| **01** | Geometry of $\mathbb{R}^n$ | Feature space geometry; level sets of the log-likelihood; decision boundary as a hyperplane |
| **02** | Partial Derivatives & Differentiability | Partial derivatives of the log-likelihood; the chain rule applied to $\sigma(\boldsymbol{w}^{\top}\boldsymbol{x})$ |
| **03** | The Gradient & Directional Derivatives | $\nabla_{\boldsymbol{w}}\log p(\boldsymbol{y}\|\boldsymbol{X},\boldsymbol{w}) = \boldsymbol{X}^{\top}(\boldsymbol{y}-\hat{\boldsymbol{p}})$; chain rule on compositions for reparameterization |
| **04** | Optimization in Multiple Variables | Gradient ascent on the ELBO; convexity of the KL; learning rate selection |
| **05** | Multiple Integrals | Normalizing constant $(2\pi)^{p/2}$ of the prior $p(\boldsymbol{w})$; expectations as integrals over $q(\boldsymbol{w})$ |
| **06** | The Multivariate Normal | Prior $p(\boldsymbol{w}) = \mathcal{N}(\boldsymbol{0},\sigma_0^2\boldsymbol{I})$; KL closed form; Bayesian posterior update = Ridge MAP |
| **07** | Vector Calculus | ELBO decomposition; score field perspective on the reparameterization gradient; KL as a flux-divergence duality |
| **08** | Capstone Project | Full integration: problem setup → ELBO derivation → gradient computation → implementation → posterior analysis |

---

**The four statistical threads, completed:**

| Thread | First appearance | Resolution in Module 08 |
|---|---|---|
| `[PD]` — Probability distributions | Module 05 | Posterior predictive distribution; calibration |
| `[MLE]` — Maximum likelihood | Module 03 Unit 01 | MLE is VI with a flat prior ($\sigma_0^2 \to \infty$) |
| `[BAY]` — Bayesian inference | Module 06 Unit 02 | Full variational Bayes posterior via ELBO |
| `[GLM]` — Generalized linear models | Module 03 Unit 02 | Logistic regression likelihood in the ELBO |

In [ ]:
# ============================================================
# Final summary figure — everything in one view
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: posterior distribution over decision boundaries
ax = axes[0]
rng3 = np.random.default_rng(99)
eps3 = rng3.standard_normal((200, p_dim))
w_vis = m_star + s_star * eps3   # 200 posterior weight samples

for c, col, lab in [(0, TS_BLUE, 'Setosa'), (1, TS_AMBER, 'Versicolor')]:
    idx = y == c
    ax.scatter(X_scaled[idx,0], X_scaled[idx,1], color=col, alpha=0.7,
               s=55, edgecolors='white', linewidths=0.5, label=lab, zorder=4)

x1_line = np.linspace(-3, 3.5, 200)
for i, w in enumerate(w_vis):
    if abs(w[2]) > 1e-6:
        x2_line = -(w[0] + w[1]*x1_line) / w[2]
        mask = (x2_line > -3) & (x2_line < 3)
        ax.plot(x1_line[mask], x2_line[mask],
                color=TS_GREEN, alpha=0.05, lw=0.8)

# Posterior mean boundary (solid)
if abs(m_star[2]) > 1e-6:
    x2_mean = -(m_star[0] + m_star[1]*x1_line) / m_star[2]
    mask_m = (x2_mean > -3) & (x2_mean < 3)
    ax.plot(x1_line[mask_m], x2_mean[mask_m], color=TS_RED, lw=2.5,
            label='Posterior mean boundary', zorder=5)

ax.set_xlim(-3, 3.5); ax.set_ylim(-3, 3)
ax.set_xlabel('Sepal length (std)'); ax.set_ylabel('Sepal width (std)')
ax.set_title('200 posterior decision boundaries\n(green) + mean (red)')
ax.legend(fontsize=8)
ax.set_aspect('equal')

# Right: ELBO and KL components over training
ax2 = axes[1]
results_full = run_vi(X, y, sigma0_sq=1.0, lr=0.05, n_iter=600, K=20, seed=42)

# Recompute KL and expected LL at each checkpoint for plotting
n_check = 60
check_iters = np.linspace(0, 599, n_check, dtype=int)

ax2.plot(results_full['elbo'], color=TS_BLUE, lw=2, label='ELBO', alpha=0.9)
ax2.set_xlabel('Iteration'); ax2.set_ylabel('Nats')
ax2.set_title('ELBO convergence\n(Bayesian logistic regression, Iris)')
ax2.axhline(results_full['elbo'][-1], color=TS_AMBER, lw=1.5, ls='--',
            label=f'Final ELBO = {results_full["elbo"][-1]:.1f}')
ax2.legend(fontsize=9)

plt.suptitle('Module 08 Capstone — Bayesian Logistic Regression on Iris',
             fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

**What do you see?**

- **Left**: The 200 posterior decision boundaries (green, semi-transparent) show the full uncertainty in the classifier. They are spread across a region rather than concentrated at a single line — reflecting genuine uncertainty about the optimal classifier given the data and prior. The posterior mean boundary (red) cuts cleanly between the two classes.
- **Right**: The ELBO increases from its initial value (where $q = p$, so ELBO = expected log-likelihood under the prior) to a stable convergence level. The stochastic noise from $K=20$ samples is visible but manageable.

**The spread of the green lines quantifies everything the prior, likelihood, and data say about where the decision boundary should be.** This is what Bayesian inference adds over point estimation: not a single answer, but a calibrated distribution of answers weighted by their posterior plausibility.

In [ ]:
# Extension workspace — open-ended capstone extensions
#
# 1. ALL THREE CLASSES: Extend to Setosa vs Versicolor vs Virginica
#    using a softmax likelihood (multinomial logistic regression).
#    How does the ELBO derivation change? (Hint: there are now 3×p parameters
#    and the KL formula generalizes component-wise.)
#
# 2. ALL FOUR FEATURES: Use sepal length, sepal width, petal length, petal width.
#    The boundary is now a hyperplane in R⁴. How does classification accuracy
#    change? Does the posterior uncertainty decrease with more features?
#
# 3. PRIOR SENSITIVITY: Run VI with σ₀² ∈ {0.1, 0.5, 1.0, 5.0, 100.0}.
#    For each, record the ELBO, the posterior mean, and the classification
#    accuracy. At what σ₀² does the posterior mean approach the MLE?
#    This is the bias-variance trade-off expressed as a prior sensitivity analysis.
#
# 4. FULL COVARIANCE: Replace the diagonal variational posterior with a
#    full-rank covariance q(w) = N(m, LLᵀ) where L is a lower triangular
#    Cholesky factor. Reparameterize as w = m + L ε. Does the ELBO
#    converge higher? Do the posterior boundaries become more constrained?


---

## Final Summary — Module 08

| Phase | Unit | What was built |
|---|---|---|
| Problem Setup | 01 | Prior, likelihood, variational family, ELBO, KL closed form |
| Derivation | 02 | Reparameterization trick, ELBO gradients, gradient ascent |
| Analysis | 03 | Posterior predictive, calibration, decision boundary, retrospective |

---

## Course Complete

| Module | Title | What it built |
|---|---|---|
| 00 | Orientation & Prerequisites | Vectors, dot products, single-variable calculus |
| 01 | Geometry of $\mathbb{R}^n$ | Scalar fields, level sets, geometric intuition |
| 02 | Partial Derivatives & Differentiability | Partials, chain rule, total differential |
| 03 | The Gradient & Directional Derivatives | Gradient, Jacobian, steepest ascent, logistic regression |
| 04 | Optimization | Critical points, Hessian, convexity, Ridge, Lagrange multipliers |
| 05 | Multiple Integrals | Fubini, change of variables, $\sqrt{2\pi}$, bivariate normal |
| 06 | The Multivariate Normal | MVN density, marginals, conditionals, MLE, Fisher information |
| 07 | Vector Calculus | Divergence, curl, conservative fields, major theorems, ELBO |
| 08 | Capstone Project | Bayesian logistic regression end-to-end |

---

> *"The purpose of this course was to show that the mathematics used to train, interpret, and audit machine learning models is not a black box. Every gradient descent step, every covariance matrix, every KL divergence, every confidence interval has a precise mathematical meaning — and that meaning is accessible to anyone who has worked through these nine modules. You now have the tools."*

---
*Module 08, Unit 03 — Threat Surfaces | fischer³ Education*